In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

# CELL 1: Environment Setup


!pip install -q segmentation-models-pytorch==0.3.3
!pip install -q albumentations==1.3.1
!pip install -q shapely

import os
import json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

# Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Utils
from sklearn.model_selection import train_test_split
from collections import defaultdict, Counter
import shutil
from shapely.geometry import Polygon
from shapely import wkt

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

print("="*60)
print("✅ ENVIRONMENT READY")
print("="*60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("="*60)


In [ ]:
class Config:
    """Project Configuration"""
    
    # === PATHS ===
    BASE_PATH = Path('/kaggle/input/xbd-dataset')
    OUTPUT_DIR = Path('/kaggle/working/xbd_processed')
    
    # Create output structure
    TRAIN_IMG_DIR = OUTPUT_DIR / 'train' / 'images'
    TRAIN_MASK_DIR = OUTPUT_DIR / 'train' / 'masks'
    VAL_IMG_DIR = OUTPUT_DIR / 'val' / 'images'
    VAL_MASK_DIR = OUTPUT_DIR / 'val' / 'masks'
    
    for d in [TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    
    # === DISASTER SELECTION ===
    DISASTER_TYPES = {
        'hurricane': ['hurricane-harvey', 'hurricane-michael', 
                     'hurricane-florence', 'hurricane-matthew'],
        'earthquake': ['guatemala-volcano', 'palu-tsunami', 
                      'mexico-earthquake'],
        'wildfire': ['woolsey-fire', 'santa-rosa-wildfire', 'socal-fire'],
        'flood': ['midwest-flooding', 'nepal-flooding']
    }
    
    # === DATASET PARAMETERS ===
    SUBSET_SIZE = 10000        # Number of images to use
    VAL_SPLIT = 0.2           # 20% validation
    IMAGE_SIZE = 512          # Resize to 512x512 (from 1024x1024)
    
    # === DAMAGE CLASSES (Aligned with your screenshot) ===
    DAMAGE_MAP = {
        'no-damage': 0,
        'minor-damage': 1,
        'major-damage': 2,
        'destroyed': 3,
        'un-classified': 0  # Treat as no-damage
    }
    NUM_CLASSES = 4
    
    # === TRAINING PARAMS ===
    BATCH_SIZE = 8
    NUM_WORKERS = 2
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # === FEATURE DIMENSIONS (Aligned with screenshot) ===
    F_SAT_DIM = 512      # Satellite embedding dimension
    F_REGION_DIM = 128   # Regional statistics dimension

config = Config()

# Explore dataset structure
print("\n" + "="*60)
print("📂 EXPLORING DATASET STRUCTURE")
print("="*60)

if not config.BASE_PATH.exists():
    print("❌ Dataset not found at:", config.BASE_PATH)
    print("\nAvailable paths:")
    for p in Path('/kaggle/input').iterdir():
        print(f"  - {p}")
else:
    print(f"✅ Dataset found at: {config.BASE_PATH}\n")
    
    # Show directory structure
    print("📁 Directory Structure:")
    for item in sorted(config.BASE_PATH.rglob('*'))[:20]:
        if item.is_dir():
            rel_path = item.relative_to(config.BASE_PATH)
            num_files = len(list(item.glob('*')))
            print(f"  {rel_path}/ ({num_files} files)")
    
    # Count total files
    all_pngs = list(config.BASE_PATH.rglob('*.png'))
    all_jsons = list(config.BASE_PATH.rglob('*.json'))
    
    print(f"\n📊 File Counts:")
    print(f"  Total PNG images: {len(all_pngs)}")
    print(f"  Total JSON labels: {len(all_jsons)}")
    
    # Show sample paths
    if all_pngs:
        print(f"\n📸 Sample Image: {all_pngs[0].name}")
    if all_jsons:
        print(f"📄 Sample Label: {all_jsons[0].name}")

print("\n" + "="*60)
print("⚙️ CONFIGURATION SUMMARY")
print("="*60)
print(f"Subset Size: {config.SUBSET_SIZE} images")
print(f"Image Size: {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")
print(f"Damage Classes: {config.NUM_CLASSES}")
print(f"Output: F_sat({config.F_SAT_DIM}), P_x(4), F_region({config.F_REGION_DIM})")
print(f"Device: {config.DEVICE}")
print("="*60 + "\n")

In [ ]:

# CELL 3: Load and Filter Dataset


class xBDDatasetLoader:
    """Load and filter xBD dataset"""
    
    def __init__(self, base_path):
        self.base_path = Path(base_path)
        self.data_pairs = []
    
    def find_post_disaster_pairs(self):
        """Find post-disaster images with matching JSON labels"""
        
        print("🔍 Searching for post-disaster image-label pairs...\n")
        
        # Search for all post-disaster images
        post_images = list(self.base_path.rglob('*_post_disaster.png'))
        
        print(f"Found {len(post_images)} post-disaster images")
        
        # Match with JSON labels
        for img_path in tqdm(post_images, desc="Matching labels"):
            # Extract base ID (remove _post_disaster.png)
            img_name = img_path.stem  # e.g., "hurricane-harvey_00000001_post_disaster"
            base_id = img_name.replace('_post_disaster', '')
            
            # Find corresponding JSON label
            json_path = img_path.parent.parent / 'labels' / f"{base_id}_post_disaster.json"
            
            # Alternative: look in same directory
            if not json_path.exists():
                json_path = img_path.parent / f"{base_id}_post_disaster.json"
            
            # Alternative: look for any matching JSON
            if not json_path.exists():
                possible_jsons = list(img_path.parent.parent.rglob(f"*{base_id}*.json"))
                if possible_jsons:
                    json_path = possible_jsons[0]
            
            if json_path.exists():
                # Extract disaster name from path
                disaster = self._extract_disaster(img_path)
                
                self.data_pairs.append({
                    'image_path': str(img_path),
                    'label_path': str(json_path),
                    'image_id': base_id,
                    'disaster': disaster
                })
        
        print(f"✅ Successfully paired: {len(self.data_pairs)} samples\n")
        return self.data_pairs
    
    def _extract_disaster(self, path):
        """Extract disaster type from path"""
        path_str = str(path).lower()
        
        # Check each disaster type
        for category, disasters in config.DISASTER_TYPES.items():
            for disaster in disasters:
                if disaster in path_str:
                    return disaster
        
        # Fallback: extract from filename
        parts = Path(path).stem.split('_')
        if len(parts) > 0:
            return parts[0]
        
        return 'unknown'
    
    def filter_by_disasters(self, selected_disasters):
        """Filter dataset by selected disaster types"""
        
        filtered = [
            pair for pair in self.data_pairs 
            if pair['disaster'] in selected_disasters
        ]
        
        print(f"📊 Filtered to {len(filtered)} images from selected disasters")
        return filtered
    
    def get_statistics(self, data_pairs):
        """Get dataset statistics"""
        
        disaster_counts = Counter([p['disaster'] for p in data_pairs])
        
        print("\n" + "="*60)
        print("📈 DATASET STATISTICS")
        print("="*60)
        
        for disaster, count in disaster_counts.most_common():
            category = 'unknown'
            for cat, disasters in config.DISASTER_TYPES.items():
                if disaster in disasters:
                    category = cat
                    break
            print(f"  {disaster:30s} [{category:10s}]: {count:4d} images")
        
        print("="*60 + "\n")
        
        return disaster_counts

# Load dataset
loader = xBDDatasetLoader(config.BASE_PATH)
all_pairs = loader.find_post_disaster_pairs()

# Flatten selected disasters list
selected_disasters = [d for disasters in config.DISASTER_TYPES.values() 
                     for d in disasters]

# Filter by disaster types
filtered_pairs = loader.filter_by_disasters(selected_disasters)

# Show statistics
stats = loader.get_statistics(filtered_pairs)

# Sample subset if needed
if len(filtered_pairs) > config.SUBSET_SIZE:
    print(f"✂️ Sampling {config.SUBSET_SIZE} from {len(filtered_pairs)} total images")
    print("   Using stratified sampling to maintain disaster balance...\n")
    
    # Convert to DataFrame for stratified sampling
    df = pd.DataFrame(filtered_pairs)
    
    # Calculate samples per disaster (proportional)
    samples_per_disaster = {}
    total = len(filtered_pairs)
    for disaster in df['disaster'].unique():
        proportion = len(df[df['disaster'] == disaster]) / total
        samples_per_disaster[disaster] = int(config.SUBSET_SIZE * proportion)
    
    # Sample
    sampled_dfs = []
    for disaster, n_samples in samples_per_disaster.items():
        disaster_df = df[df['disaster'] == disaster]
        if len(disaster_df) >= n_samples:
            sampled_dfs.append(disaster_df.sample(n_samples, random_state=42))
        else:
            sampled_dfs.append(disaster_df)
    
    sampled_df = pd.concat(sampled_dfs, ignore_index=True)
    data_pairs = sampled_df.to_dict('records')
    
    print(f"✅ Final dataset: {len(data_pairs)} images")
    loader.get_statistics(data_pairs)
else:
    data_pairs = filtered_pairs

# Save metadata
metadata_df = pd.DataFrame(data_pairs)
metadata_df.to_csv(config.OUTPUT_DIR / 'raw_metadata.csv', index=False)

print(f"💾 Raw metadata saved: {config.OUTPUT_DIR / 'raw_metadata.csv'}")


In [ ]:

# CELL 4: JSON to Segmentation Mask Converter


class JSONToMaskConverter:
    """Convert xBD JSON annotations to pixel-wise damage masks"""
    
    def __init__(self, image_size=512):
        self.image_size = image_size
        self.damage_map = config.DAMAGE_MAP
        
    def json_to_mask(self, json_path, original_shape):
        """
        Convert JSON polygons to damage segmentation mask
        
        Returns:
            mask: np.array [H, W] with values 0-3 (damage classes)
        """
        
        # Read JSON
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        # Initialize mask with no-damage (class 0)
        mask = np.zeros(original_shape[:2], dtype=np.uint8)
        
        # Check for features
        if 'features' not in data or 'xy' not in data['features']:
            return mask
        
        features = data['features']['xy']
        
        # Process each building polygon
        for feature in features:
            try:
                # Extract polygon coordinates
                if 'wkt' in feature:
                    # Parse WKT format: "POLYGON ((x1 y1, x2 y2, ...))"
                    poly = wkt.loads(feature['wkt'])
                    coords = np.array(poly.exterior.coords, dtype=np.int32)
                
                elif 'coordinates' in feature:
                    coords = np.array(feature['coordinates'], dtype=np.int32)
                
                else:
                    continue
                
                # Get damage classification
                props = feature.get('properties', {})
                damage_type = props.get('subtype', 'no-damage')
                
                # Map to class index (0-3)
                damage_class = self.damage_map.get(damage_type, 0)
                
                # Draw filled polygon on mask
                cv2.fillPoly(mask, [coords], int(damage_class))
                
            except Exception as e:
                # Skip problematic polygons
                continue
        
        return mask
    
    def visualize_sample(self, image, mask, title="Sample"):
        """Visualize image and mask"""
        
        # Damage class names
        class_names = ['No Damage', 'Minor', 'Major', 'Destroyed']
        colors = ['green', 'yellow', 'orange', 'red']
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Original image
        axes[0].imshow(image)
        axes[0].set_title(f'{title}\nPost-Disaster Image', fontsize=12)
        axes[0].axis('off')
        
        # Damage mask
        im = axes[1].imshow(mask, cmap='RdYlGn_r', vmin=0, vmax=3)
        axes[1].set_title('Damage Segmentation Mask', fontsize=12)
        axes[1].axis('off')
        
        # Overlay
        axes[2].imshow(image)
        axes[2].imshow(mask, alpha=0.6, cmap='RdYlGn_r', vmin=0, vmax=3)
        axes[2].set_title('Overlay (Image + Mask)', fontsize=12)
        axes[2].axis('off')
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1], orientation='horizontal', 
                           pad=0.05, fraction=0.046)
        cbar.set_ticks([0, 1, 2, 3])
        cbar.set_ticklabels(class_names)
        
        # Show pixel statistics
        unique, counts = np.unique(mask, return_counts=True)
        stats_text = "Pixel Distribution:\n"
        for cls, count in zip(unique, counts):
            pct = (count / mask.size) * 100
            stats_text += f"{class_names[cls]}: {pct:.1f}%\n"
        
        plt.figtext(0.99, 0.5, stats_text, fontsize=10, 
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                   verticalalignment='center', horizontalalignment='left')
        
        plt.tight_layout()
        plt.show()

# Initialize converter
mask_converter = JSONToMaskConverter(image_size=config.IMAGE_SIZE)

print("="*60)
print("✅ MASK CONVERTER INITIALIZED")
print("="*60)
print(f"Target Size: {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")
print(f"Damage Classes: {config.DAMAGE_MAP}")
print("="*60 + "\n")


In [ ]:

# CELL 5: Image Preprocessing


class ImagePreprocessor:
    """Preprocess images for DeepLabV3+"""
    
    def __init__(self, target_size=512):
        self.target_size = (target_size, target_size)
        
        # ImageNet normalization (required for pre-trained models)
        self.mean = np.array([0.485, 0.456, 0.406])
        self.std = np.array([0.229, 0.224, 0.225])
    
    def load_and_resize(self, image_path):
        """Load and resize image from 1024x1024 to target size"""
        
        # Read image
        image = cv2.imread(str(image_path))
        if image is None:
            raise ValueError(f"Failed to load: {image_path}")
        
        # BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Resize using high-quality interpolation
        image = cv2.resize(image, self.target_size, 
                          interpolation=cv2.INTER_LINEAR)
        
        return image
    
    def resize_mask(self, mask, target_size=None):
        """Resize mask using nearest neighbor (preserves class labels)"""
        
        if target_size is None:
            target_size = self.target_size
        
        # INTER_NEAREST ensures class labels remain intact
        mask = cv2.resize(mask, target_size, 
                         interpolation=cv2.INTER_NEAREST)
        
        return mask
    
    def validate_pair(self, image, mask):
        """Quality check for image-mask pair"""
        
        # Check shapes match
        if image.shape[:2] != mask.shape[:2]:
            return False, f"Shape mismatch: {image.shape[:2]} vs {mask.shape[:2]}"
        
        # Check valid damage classes
        unique_classes = np.unique(mask)
        valid_classes = set(range(config.NUM_CLASSES))
        
        if not set(unique_classes).issubset(valid_classes):
            return False, f"Invalid classes: {unique_classes}"
        
        # Check image has content (not blank)
        if image.std() < 5:
            return False, "Low variance (blank image)"
        
        # Check mask has some damage annotations
        if mask.max() == 0 and mask.min() == 0:
            return False, "Empty mask (no buildings)"
        
        return True, "OK"
    
    def get_class_distribution(self, mask):
        """Get pixel count per damage class"""
        
        unique, counts = np.unique(mask, return_counts=True)
        dist = {int(cls): int(count) for cls, count in zip(unique, counts)}
        
        # Ensure all classes are represented
        for cls in range(config.NUM_CLASSES):
            if cls not in dist:
                dist[cls] = 0
        
        return dist

# Initialize preprocessor
preprocessor = ImagePreprocessor(target_size=config.IMAGE_SIZE)

print("="*60)
print("✅ PREPROCESSOR INITIALIZED")
print("="*60)
print(f"Input Size: 1024x1024 (original xBD)")
print(f"Output Size: {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")
print(f"Interpolation: LINEAR (image), NEAREST (mask)")
print(f"Normalization: ImageNet mean/std")
print("="*60 + "\n")


In [ ]:

# CELL 6: Process Dataset (Images + Masks)


def process_dataset(data_pairs, preprocessor, converter, visualize_samples=3):
    """
    Process all image-label pairs:
    1. Load post-disaster image
    2. Convert JSON to damage mask
    3. Resize both to 512x512
    4. Quality check
    5. Save processed data
    """
    
    print("="*60)
    print(f"🔄 PROCESSING {len(data_pairs)} IMAGE-MASK PAIRS")
    print("="*60 + "\n")
    
    processed_data = []
    failed_samples = []
    global_class_stats = {0: 0, 1: 0, 2: 0, 3: 0}
    
    for idx, pair in enumerate(tqdm(data_pairs, desc="Processing")):
        try:
            # Load and resize image
            image = preprocessor.load_and_resize(pair['image_path'])
            
            # Create mask from JSON (at original size first)
            original_img = cv2.imread(pair['image_path'])
            mask_full = converter.json_to_mask(pair['label_path'], 
                                               original_img.shape)
            
            # Resize mask to match image
            mask = preprocessor.resize_mask(mask_full)
            
            # Validate
            is_valid, message = preprocessor.validate_pair(image, mask)
            if not is_valid:
                failed_samples.append((pair['image_id'], message))
                continue
            
            # Get class distribution
            class_dist = preprocessor.get_class_distribution(mask)
            
            # Update global statistics
            for cls, count in class_dist.items():
                global_class_stats[cls] += count
            
            # Save files
            image_filename = f"{pair['image_id']}.png"
            mask_filename = f"{pair['image_id']}_mask.png"
            
            # Save to working directory (temporary)
            img_save_path = config.OUTPUT_DIR / image_filename
            mask_save_path = config.OUTPUT_DIR / mask_filename
            
            cv2.imwrite(str(img_save_path), 
                       cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(mask_save_path), mask)
            
            # Record metadata
            processed_data.append({
                'image_id': pair['image_id'],
                'image_file': image_filename,
                'mask_file': mask_filename,
                'disaster': pair['disaster'],
                'image_shape': f"{image.shape[0]}x{image.shape[1]}",
                'class_0_pixels': class_dist[0],
                'class_1_pixels': class_dist[1],
                'class_2_pixels': class_dist[2],
                'class_3_pixels': class_dist[3]
            })
            
            # Visualize first N samples
            if idx < visualize_samples:
                print(f"\n{'='*60}")
                print(f"📸 SAMPLE {idx+1}: {pair['disaster']}")
                print(f"{'='*60}")
                converter.visualize_sample(image, mask, 
                                          title=f"Sample {idx+1}")
        
        except Exception as e:
            failed_samples.append((pair['image_id'], str(e)))
            continue
    
    # Results summary
    print(f"\n{'='*60}")
    print("📊 PROCESSING RESULTS")
    print("="*60)
    print(f"✅ Successfully processed: {len(processed_data)} samples")
    print(f"❌ Failed: {len(failed_samples)} samples")
    
    if failed_samples:
        print(f"\n⚠️ Sample failures (showing first 5):")
        for img_id, reason in failed_samples[:5]:
            print(f"  {img_id}: {reason}")
    
    # Class distribution
    total_pixels = sum(global_class_stats.values())
    class_names = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
    
    print(f"\n📊 GLOBAL CLASS DISTRIBUTION:")
    print("-"*60)
    for cls in range(4):
        count = global_class_stats[cls]
        pct = (count / total_pixels) * 100 if total_pixels > 0 else 0
        print(f"  Class {cls} ({class_names[cls]:15s}): "
              f"{count:12,} pixels ({pct:5.2f}%)")
    print("-"*60 + "\n")
    
    return pd.DataFrame(processed_data), global_class_stats

# Process all data
processed_df, class_stats = process_dataset(
    data_pairs, 
    preprocessor, 
    mask_converter,
    visualize_samples=3
)

# Save metadata
metadata_path = config.OUTPUT_DIR / 'processed_metadata.csv'
processed_df.to_csv(metadata_path, index=False)

print(f"💾 Processed metadata saved: {metadata_path}")
print(f"📁 Total files created: {len(processed_df) * 2} (images + masks)\n")


In [ ]:

# CELL 7: Create Train/Val Split


def create_train_val_split(processed_df, val_split=0.2):
    """
    Split dataset into train/val with:
    - Stratification by disaster type
    - Organized directory structure
    """
    
    print("="*60)
    print(f"📂 CREATING TRAIN/VAL SPLIT")
    print("="*60 + "\n")
    
    # Stratified split
    train_df, val_df = train_test_split(
        processed_df,
        test_size=val_split,
        stratify=processed_df['disaster'],
        random_state=42
    )
    
    print(f"Train set: {len(train_df)} samples ({(1-val_split)*100:.0f}%)")
    print(f"Val set:   {len(val_df)} samples ({val_split*100:.0f}%)")
    
    # Move files to respective directories
    for split_name, df, img_dir, mask_dir in [
        ('TRAIN', train_df, config.TRAIN_IMG_DIR, config.TRAIN_MASK_DIR),
        ('VAL', val_df, config.VAL_IMG_DIR, config.VAL_MASK_DIR)
    ]:
        print(f"\n📦 Organizing {split_name} set...")
        
        for _, row in tqdm(df.iterrows(), total=len(df), 
                          desc=f"Moving {split_name}"):
            # Copy image
            src_img = config.OUTPUT_DIR / row['image_file']
            dst_img = img_dir / row['image_file']
            shutil.copy(str(src_img), str(dst_img))
            
            # Copy mask
            src_mask = config.OUTPUT_DIR / row['mask_file']
            dst_mask = mask_dir / row['mask_file']
            shutil.copy(str(src_mask), str(dst_mask))
    
    # Save split metadata
    train_df.to_csv(config.OUTPUT_DIR / 'train_metadata.csv', index=False)
    val_df.to_csv(config.OUTPUT_DIR / 'val_metadata.csv', index=False)
    
    # Show disaster distribution
    print(f"\n{'='*60}")
    print("📊 DISASTER DISTRIBUTION PER SPLIT")
    print("="*60)
    
    print("\n🔹 TRAIN SET:")
    train_dist = train_df['disaster'].value_counts()
    for disaster, count in train_dist.items():
        pct = (count / len(train_df)) * 100
        print(f"  {disaster:30s}: {count:4d} ({pct:5.2f}%)")
    
    print("\n🔹 VALIDATION SET:")
    val_dist = val_df['disaster'].value_counts()
    for disaster, count in val_dist.items():
        pct = (count / len(val_df)) * 100
        print(f"  {disaster:30s}: {count:4d} ({pct:5.2f}%)")
    
    print("="*60 + "\n")
    
    return train_df, val_df

# Create split
train_df, val_df = create_train_val_split(processed_df, 
                                           val_split=config.VAL_SPLIT)

# Cleanup temporary files
print("🧹 Cleaning up temporary files...")
for file in config.OUTPUT_DIR.glob('*.png'):
    file.unlink()

print("✅ Cleanup complete!\n")


In [ ]:

# CELL 8: Calculate Class Weights


def calculate_class_weights_optimized(train_df, method='median_frequency'):
    """
    Calculate class weights to handle imbalanced data
    
    Methods:
    - 'inverse_frequency': 1 / freq
    - 'median_frequency': median_freq / freq (recommended)
    """
    
    # Get pixel counts per class
    class_totals = {}
    for cls in range(config.NUM_CLASSES):
        col_name = f'class_{cls}_pixels'
        class_totals[cls] = train_df[col_name].sum()
    
    total_pixels = sum(class_totals.values())
    
    if method == 'median_frequency':
        # Median frequency balancing (better for extreme imbalance)
        frequencies = [class_totals[cls] / total_pixels 
                      for cls in range(config.NUM_CLASSES)]
        median_freq = np.median(frequencies)
        
        weights = {}
        for cls in range(config.NUM_CLASSES):
            freq = frequencies[cls]
            if freq > 0:
                weights[cls] = median_freq / freq
            else:
                weights[cls] = 1.0
    
    else:  # inverse_frequency
        weights = {}
        for cls in range(config.NUM_CLASSES):
            count = class_totals[cls]
            if count > 0:
                weights[cls] = total_pixels / (config.NUM_CLASSES * count)
            else:
                weights[cls] = 1.0
    
    # Convert to tensor
    weight_tensor = torch.FloatTensor([weights[i] 
                                       for i in range(config.NUM_CLASSES)])
    
    # Normalize to max=1 for stability
    weight_tensor = weight_tensor / weight_tensor.max()
    
    return weight_tensor, weights

# Calculate weights
class_weights_tensor, class_weights_dict = calculate_class_weights_optimized(
    train_df, method='median_frequency'
)

print("="*60)
print("⚖️ CLASS WEIGHTS FOR LOSS FUNCTION")
print("="*60)

class_names = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']

print("\nWeights (for nn.CrossEntropyLoss):")
for cls in range(config.NUM_CLASSES):
    print(f"  Class {cls} ({class_names[cls]:15s}): {class_weights_tensor[cls]:.4f}")

print(f"\nTensor: {class_weights_tensor}")
print("\n💡 Higher weights = more emphasis during training")
print("   (destroyed buildings get highest weight due to rarity)")
print("="*60 + "\n")

# Save weights for training
weights_dict = {
    'class_weights': class_weights_tensor.tolist(),
    'class_names': class_names,
    'method': 'median_frequency'
}

with open(config.OUTPUT_DIR / 'class_weights.json', 'w') as f:
    json.dump(weights_dict, f, indent=2)

print(f"💾 Weights saved: {config.OUTPUT_DIR / 'class_weights.json'}")


In [ ]:

# CELL 9: Data Augmentation Pipeline


def get_training_augmentation():
    """
    Training augmentation pipeline aligned with disaster imagery
    """
    
    return A.Compose([
        # Geometric transforms (safe for segmentation)
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        
        A.ShiftScaleRotate(
            shift_limit=0.0625,   # ±6% shift
            scale_limit=0.1,       # ±10% zoom
            rotate_limit=15,       # ±15° rotation
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            mask_value=0,
            p=0.5
        ),
        
        # Photometric transforms (image only, not mask)
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.2,
                contrast_limit=0.2,
                p=1.0
            ),
            A.HueSaturationValue(
                hue_shift_limit=10,
                sat_shift_limit=20,
                val_shift_limit=10,
                p=1.0
            ),
            A.CLAHE(clip_limit=2.0, p=1.0),
        ], p=0.5),
        
        # Noise and blur (improves robustness)
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        ], p=0.3),
        
        # Normalization (ImageNet stats for pre-trained models)
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0
        ),
        
        ToTensorV2()
    ])

def get_validation_augmentation():
    """
    Validation: only normalization (no data augmentation)
    """
    
    return A.Compose([
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0
        ),
        ToTensorV2()
    ])

# Create transforms
train_transform = get_training_augmentation()
val_transform = get_validation_augmentation()

print("="*60)
print("🔄 DATA AUGMENTATION CONFIGURED")
print("="*60)

print("\n🔹 TRAINING AUGMENTATIONS:")
print("  ✓ Horizontal/Vertical flips (50%)")
print("  ✓ Random 90° rotations (50%)")
print("  ✓ Shift/Scale/Rotate (50%)")
print("  ✓ Brightness/Contrast/HSV (50%)")
print("  ✓ Gaussian noise/blur (30%)")
print("  ✓ ImageNet normalization (100%)")

print("\n🔹 VALIDATION:")
print("  ✓ ImageNet normalization only")

print("\n💡 Augmentations increase effective dataset size ~8x")
print("="*60 + "\n")


In [ ]:

#  Import Missing PyTorch Components


from torch.utils.data import Dataset, DataLoader


In [ ]:

# CELL 10: PyTorch Dataset & DataLoaders 


class xBDDataset(Dataset):
    """
    PyTorch Dataset for xBD damage segmentation
    
    Returns dict with:
    - 'image': Tensor [3, H, W]
    - 'mask': Tensor [H, W] with class indices (0-3)
    - 'image_id': str
    - 'disaster': str
    """
    
    def __init__(self, metadata_df, images_dir, masks_dir, transform=None):
        self.metadata = metadata_df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.transform = transform
    
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        
        # Load image
        img_path = self.images_dir / row['image_file']
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Load mask
        mask_path = self.masks_dir / row['mask_file']
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        
        # Apply augmentation
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        
        # Convert mask to tensor if not already (handles both cases)
        if not isinstance(mask, torch.Tensor):
            mask = torch.from_numpy(mask).long()
        else:
            mask = mask.long()  # Already a tensor from ToTensorV2
        
        return {
            'image': image,
            'mask': mask,
            'image_id': row['image_id'],
            'disaster': row['disaster']
        }

# Create datasets
train_dataset = xBDDataset(
    train_df,
    config.TRAIN_IMG_DIR,
    config.TRAIN_MASK_DIR,
    transform=train_transform
)

val_dataset = xBDDataset(
    val_df,
    config.VAL_IMG_DIR,
    config.VAL_MASK_DIR,
    transform=val_transform
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=True if config.DEVICE == 'cuda' else False,
    drop_last=True  # For stable batch norm
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True if config.DEVICE == 'cuda' else False
)

print("="*60)
print("✅ PYTORCH DATASETS & DATALOADERS READY")
print("="*60)
print(f"\nTrain Dataset: {len(train_dataset)} samples")
print(f"Val Dataset:   {len(val_dataset)} samples")
print(f"\nTrain Batches: {len(train_loader)}")
print(f"Val Batches:   {len(val_loader)}")
print(f"Batch Size:    {config.BATCH_SIZE}")

# Test dataset
print("\n🧪 Testing dataset loading...")
sample = train_dataset[0]
print(f"  Image shape: {sample['image'].shape}")
print(f"  Image dtype: {sample['image'].dtype}")
print(f"  Image range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")
print(f"  Mask shape:  {sample['mask'].shape}")
print(f"  Mask dtype:  {sample['mask'].dtype}")
print(f"  Mask classes: {torch.unique(sample['mask']).tolist()}")
print(f"  Image ID:    {sample['image_id']}")
print(f"  Disaster:    {sample['disaster']}")

# Test batch loading
print("\n🧪 Testing batch loading...")
batch = next(iter(train_loader))
print(f"  Batch images shape: {batch['image'].shape}")
print(f"  Batch masks shape:  {batch['mask'].shape}")

print("\n✅ All tests passed!")
print("="*60 + "\n")

In [ ]:

# CELL 11: Visualize Augmented Batch


def denormalize_image(tensor):
    """Denormalize image tensor for visualization"""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    # Denormalize
    tensor = tensor * std + mean
    
    # Clamp to [0, 1]
    tensor = torch.clamp(tensor, 0, 1)
    
    return tensor

# Get a batch
batch = next(iter(train_loader))

# Visualize
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

class_names = ['No Damage', 'Minor', 'Major', 'Destroyed']

for i in range(4):
    # Denormalize image
    img = denormalize_image(batch['image'][i])
    img = img.permute(1, 2, 0).numpy()
    
    mask = batch['mask'][i].numpy()
    disaster = batch['disaster'][i]
    
    # Row 1: Original image
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Image {i+1}\n{disaster}", fontsize=10)
    axes[0, i].axis('off')
    
    # Row 2: Damage mask
    axes[1, i].imshow(mask, cmap='RdYlGn_r', vmin=0, vmax=3)
    axes[1, i].set_title(f"Damage Mask", fontsize=10)
    axes[1, i].axis('off')
    
    # Row 3: Overlay
    axes[2, i].imshow(img)
    axes[2, i].imshow(mask, alpha=0.5, cmap='RdYlGn_r', vmin=0, vmax=3)
    axes[2, i].set_title(f"Overlay", fontsize=10)
    axes[2, i].axis('off')

plt.suptitle('Training Batch with Augmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'augmented_batch_visualization.png', 
            dpi=150, bbox_inches='tight')
plt.show()

print(f"💾 Visualization saved: {config.OUTPUT_DIR / 'augmented_batch_visualization.png'}")

In [ ]:

# CELL 12: Final Summary & Save Config


print("\n" + "="*80)
print(" " * 20 + "🎯 PREPROCESSING PIPELINE COMPLETE")
print("="*80 + "\n")

# Dataset summary
print("📊 DATASET SUMMARY")
print("-"*80)
print(f"Total Samples Processed:    {len(processed_df)}")
print(f"Train Samples:              {len(train_df)} ({len(train_df)/len(processed_df)*100:.1f}%)")
print(f"Validation Samples:         {len(val_df)} ({len(val_df)/len(processed_df)*100:.1f}%)")
print(f"Image Size:                 {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")
print(f"Number of Damage Classes:   {config.NUM_CLASSES}")
print(f"Disaster Types:             {len(train_df['disaster'].unique())}")
print("-"*80 + "\n")

# Feature outputs (aligned with your screenshot)
print("🎯 MODEL OUTPUT CONFIGURATION (Aligned with Screenshot)")
print("-"*80)
print(f"Feature              Symbol      Dim         Use")
print(f"Satellite embedding  F_sat       {config.F_SAT_DIM}         Physical damage")
print(f"Damage probs         P_x         4           Severity semantics")
print(f"Region stats (opt)   F_region    {config.F_REGION_DIM}         Spatial context")
print("-"*80 + "\n")

# Training readiness
print("🚀 TRAINING READINESS")
print("-"*80)
print(f"Batch Size:                 {config.BATCH_SIZE}")
print(f"Train Batches per Epoch:    {len(train_loader)}")
print(f"Val Batches per Epoch:      {len(val_loader)}")
print(f"Estimated Time (50 epochs): ~{len(train_loader) * 50 / 3600:.1f} hours")
print(f"GPU Memory Required:        ~{config.BATCH_SIZE * 2.5:.1f} GB")
print(f"Class Weights:              {class_weights_tensor.tolist()}")
print("-"*80 + "\n")

# Saved files
print("💾 SAVED FILES")
print("-"*80)
print(f"Train Images:     {config.TRAIN_IMG_DIR}")
print(f"Train Masks:      {config.TRAIN_MASK_DIR}")
print(f"Val Images:       {config.VAL_IMG_DIR}")
print(f"Val Masks:        {config.VAL_MASK_DIR}")
print(f"Metadata:         {config.OUTPUT_DIR}")
print("-"*80 + "\n")

# Save complete configuration
final_config = {
    'dataset': {
        'name': 'xBD',
        'source': 'kaggle:qianlanzz/xbd-dataset',
        'total_samples': len(processed_df),
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'disaster_types': train_df['disaster'].unique().tolist()
    },
    'preprocessing': {
        'original_size': '1024x1024',
        'target_size': f'{config.IMAGE_SIZE}x{config.IMAGE_SIZE}',
        'normalization': 'ImageNet',
        'augmentation': 'albumentations'
    },
    'model_output': {
        'F_sat': {'dim': config.F_SAT_DIM, 'use': 'Physical damage'},
        'P_x': {'dim': 4, 'use': 'Severity semantics'},
        'F_region': {'dim': config.F_REGION_DIM, 'use': 'Spatial context'}
    },
    'training': {
        'batch_size': config.BATCH_SIZE,
        'num_classes': config.NUM_CLASSES,
        'class_weights': class_weights_tensor.tolist(),
        'damage_classes': list(config.DAMAGE_MAP.keys())
    },
    'paths': {
        'train_images': str(config.TRAIN_IMG_DIR),
        'train_masks': str(config.TRAIN_MASK_DIR),
        'val_images': str(config.VAL_IMG_DIR),
        'val_masks': str(config.VAL_MASK_DIR)
    }
}

config_path = config.OUTPUT_DIR / 'preprocessing_config.json'
with open(config_path, 'w') as f:
    json.dump(final_config, f, indent=2)

print(f"💾 Complete configuration saved: {config_path}")

print("\n" + "="*80)
print("✅ READY FOR DEEPLABV3+ TRAINING!")
print("="*80)
print("\nNext steps:")
print("1. Build modified DeepLabV3+ model (with F_sat, P_x, F_region outputs)")
print("2. Define training loop with class-weighted loss")
print("3. Train for 50-100 epochs")
print("4. Extract features for multimodal fusion")
print("\n" + "="*80 + "\n")


In [ ]:


import torch.nn.functional as F

print("✅ torch.nn.functional imported as F")

In [ ]:

# CELL 13: Modified DeepLabV3+ Architecture


class RegionalStatsModule(nn.Module):
    """
    Extract F_region: Regional spatial context statistics
    Aligned with screenshot: F_region (K-dim)
    """
    
    def __init__(self, in_channels=256, out_dim=128):
        super().__init__()
        
        # Combine decoder features with damage probabilities
        # Note: decoder features will be upsampled to match damage_probs size
        self.conv_fusion = nn.Sequential(
            nn.Conv2d(in_channels + 4, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        
        # Spatial pooling to extract regional statistics
        self.spatial_pool = nn.AdaptiveAvgPool2d((4, 4))  # 4x4 spatial grid
        
        # FC layers to produce F_region
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, out_dim)
        )
    
    def forward(self, decoder_features, damage_probs):
        """
        Args:
            decoder_features: [B, 256, H', W'] from decoder
            damage_probs: [B, 4, H, W] damage probability maps (full resolution)
        
        Returns:
            F_region: [B, out_dim] regional statistics vector
        """
        # Upsample decoder features to match damage_probs size
        if decoder_features.shape[2:] != damage_probs.shape[2:]:
            decoder_features = F.interpolate(
                decoder_features,
                size=damage_probs.shape[2:],
                mode='bilinear',
                align_corners=False
            )
        
        # Concatenate decoder features with damage maps
        x = torch.cat([decoder_features, damage_probs], dim=1)
        
        # Process
        x = self.conv_fusion(x)
        x = self.spatial_pool(x)
        x = x.flatten(1)
        
        F_region = self.fc(x)
        
        return F_region


class DeepLabV3Plus_xBD(nn.Module):
    """
    Modified DeepLabV3+ for xBD damage assessment
    
    Outputs (aligned with screenshot):
    - F_sat: [B, 512/768] - Satellite embedding for physical damage
    - P_x: [B, 4, H, W] - Damage probability maps (severity semantics)
    - F_region: [B, K] - Regional statistics (spatial context)
    """
    
    def __init__(self, 
                 num_classes=4,
                 encoder_name='resnet101',
                 encoder_weights='imagenet',
                 F_sat_dim=512,
                 F_region_dim=128):
        
        super().__init__()
        
        self.num_classes = num_classes
        self.F_sat_dim = F_sat_dim
        
        # Base DeepLabV3+ model
        self.base_model = smp.DeepLabV3Plus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            classes=num_classes,
            activation=None  # We'll apply softmax separately
        )
        
        # Extract encoder output channels (ResNet101: 2048)
        if 'resnet101' in encoder_name:
            encoder_channels = 2048
        elif 'resnet50' in encoder_name:
            encoder_channels = 2048
        else:
            encoder_channels = 2048
        
        # F_sat: Satellite embedding extractor
        # Extract from deep encoder features before decoder
        self.embedding_extractor = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # Global average pooling
            nn.Flatten(),
            nn.Linear(encoder_channels, F_sat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )
        
        # F_region: Regional statistics module
        self.regional_stats = RegionalStatsModule(
            in_channels=256,  # Decoder output channels
            out_dim=F_region_dim
        )
    
    def forward(self, x, return_features=True):
        """
        Args:
            x: [B, 3, H, W] input images
            return_features: If True, return F_sat and F_region
        
        Returns:
            dict with:
                - 'logits': [B, 4, H, W] raw scores for loss
                - 'P_x': [B, 4, H, W] damage probabilities
                - 'F_sat': [B, 512] satellite embeddings (if return_features)
                - 'F_region': [B, K] regional stats (if return_features)
        """
        
        # Get encoder features
        features = self.base_model.encoder(x)
        
        # Extract F_sat from deepest encoder features
        if return_features:
            F_sat = self.embedding_extractor(features[-1])
        
        # Decoder
        decoder_output = self.base_model.decoder(*features)
        
        # Segmentation head - get logits
        logits = self.base_model.segmentation_head(decoder_output)
        
        # P_x: Damage probabilities
        P_x = torch.softmax(logits, dim=1)
        
        # F_region: Regional statistics (uses decoder_output before upsampling)
        if return_features:
            F_region = self.regional_stats(decoder_output, P_x)
        
        # Return outputs
        output = {
            'logits': logits,  # For training loss
            'P_x': P_x,        # Damage probabilities
        }
        
        if return_features:
            output['F_sat'] = F_sat
            output['F_region'] = F_region
        
        return output


# Initialize model
print("="*60)
print("🏗️ BUILDING MODIFIED DEEPLABV3+ MODEL")
print("="*60)

model = DeepLabV3Plus_xBD(
    num_classes=config.NUM_CLASSES,
    encoder_name='resnet101',
    encoder_weights='imagenet',
    F_sat_dim=config.F_SAT_DIM,
    F_region_dim=config.F_REGION_DIM
)

model = model.to(config.DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Statistics:")
print(f"  Encoder: ResNet101 (ImageNet pretrained)")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1e6:.1f} MB")

print(f"\n🎯 Output Features (Aligned with Screenshot):")
print(f"  F_sat:    [{config.BATCH_SIZE}, {config.F_SAT_DIM}]       - Satellite embedding (physical damage)")
print(f"  P_x:      [{config.BATCH_SIZE}, 4, 512, 512] - Damage probabilities (severity semantics)")
print(f"  F_region: [{config.BATCH_SIZE}, {config.F_REGION_DIM}]       - Regional statistics (spatial context)")

# Test forward pass
print(f"\n🧪 Testing forward pass...")
with torch.no_grad():
    dummy_input = torch.randn(2, 3, 512, 512).to(config.DEVICE)
    outputs = model(dummy_input, return_features=True)
    
    print(f"  ✓ Logits shape: {outputs['logits'].shape}")
    print(f"  ✓ P_x shape: {outputs['P_x'].shape}")
    print(f"  ✓ F_sat shape: {outputs['F_sat'].shape}")
    print(f"  ✓ F_region shape: {outputs['F_region'].shape}")

print("\n✅ Model initialized successfully!")
print("="*60 + "\n")

In [ ]:
# CELL 14: Loss, Optimizer, Scheduler (IMPROVED)
# ================================================
# KEY FIXES:
#   1. Dice + Focal combined loss (handles class imbalance)
#   2. Higher encoder LR (3e-5) for satellite domain adaptation
#   3. Warmup + Cosine schedule (prevents early instability)

import torch.nn.functional as F_func

# --- Dice Loss ---
class DiceLoss(nn.Module):
    """Soft Dice — handles extreme class imbalance in segmentation."""
    def __init__(self, num_classes=4, smooth=1.0, class_weights=None):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.class_weights = class_weights

    def forward(self, logits, targets):
        probs = F_func.softmax(logits, dim=1)
        targets_oh = F_func.one_hot(targets.long(), self.num_classes).permute(0,3,1,2).float()
        dims = (0, 2, 3)
        inter = (probs * targets_oh).sum(dim=dims)
        card = probs.sum(dim=dims) + targets_oh.sum(dim=dims)
        dice = (2.0 * inter + self.smooth) / (card + self.smooth)
        if self.class_weights is not None:
            w = self.class_weights.to(dice.device)
            return 1.0 - (dice * w).sum() / w.sum()
        return 1.0 - dice.mean()

# --- Focal Loss ---
class FocalLoss(nn.Module):
    """Down-weights easy (no-damage) pixels, focuses on hard damage pixels."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F_func.cross_entropy(logits, targets.long(), reduction="none")
        pt = torch.exp(-ce)
        fl = ((1 - pt) ** self.gamma) * ce
        if self.alpha is not None:
            fl = fl * self.alpha.to(logits.device)[targets.long()]
        return fl.mean()

# --- Combined Loss ---
class CombinedSegLoss(nn.Module):
    def __init__(self, num_classes=4, class_weights=None, dice_w=0.5, focal_w=0.5, gamma=2.0):
        super().__init__()
        self.dice = DiceLoss(num_classes, class_weights=class_weights)
        self.focal = FocalLoss(alpha=class_weights, gamma=gamma)
        self.dice_w = dice_w
        self.focal_w = focal_w

    def forward(self, logits, targets):
        return self.dice_w * self.dice(logits, targets) + self.focal_w * self.focal(logits, targets)


# Recompute class weights using effective number of samples
import numpy as _np
beta = 0.999
class_pixels = []
for cls in range(config.NUM_CLASSES):
    class_pixels.append(train_df[f"class_{cls}_pixels"].sum())
class_pixels = _np.array(class_pixels, dtype=_np.float64)
eff = (1.0 - _np.power(beta, class_pixels)) / (1.0 - beta)
weights = 1.0 / (eff + 1e-8)
weights = weights / weights.min()
class_weights_tensor = torch.FloatTensor(weights)

print("Effective-samples class weights:", class_weights_tensor.tolist())

# Loss
criterion = CombinedSegLoss(
    num_classes=config.NUM_CLASSES,
    class_weights=class_weights_tensor,
    dice_w=0.5, focal_w=0.5, gamma=2.0,
)

# Optimizer — higher encoder LR for satellite domain
optimizer = torch.optim.AdamW([
    {'params': model.base_model.encoder.parameters(), 'lr': 3e-5},   # was 1e-5
    {'params': model.base_model.decoder.parameters(), 'lr': 1e-4},   # was 5e-5
    {'params': model.base_model.segmentation_head.parameters(), 'lr': 2e-4},
    {'params': model.embedding_extractor.parameters(), 'lr': 2e-4},
    {'params': model.regional_stats.parameters(), 'lr': 2e-4},
], weight_decay=1e-4)

# LR scheduler: 5-epoch warmup + cosine decay
from torch.optim.lr_scheduler import _LRScheduler

class WarmupCosineScheduler(_LRScheduler):
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-7, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        if self.last_epoch < self.warmup_epochs:
            factor = (self.last_epoch + 1) / self.warmup_epochs
            return [base_lr * factor for base_lr in self.base_lrs]
        progress = (self.last_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
        factor = 0.5 * (1.0 + np.cos(np.pi * progress))
        return [self.min_lr + (base_lr - self.min_lr) * factor for base_lr in self.base_lrs]

num_epochs = 60
scheduler = WarmupCosineScheduler(optimizer, warmup_epochs=5, total_epochs=num_epochs, min_lr=1e-7)

print("\n" + "="*60)
print("IMPROVED TRAINING CONFIGURATION")
print("="*60)
print(f"Loss: Dice (0.5) + Focal (0.5, gamma=2.0)")
print(f"Optimizer: AdamW (encoder 3e-5, decoder 1e-4, heads 2e-4)")
print(f"Scheduler: Warmup(5) + Cosine -> 1e-7")
print(f"Epochs: {num_epochs} (early stop patience=15)")
print(f"Class weights: {class_weights_tensor.tolist()}")
print("="*60)


In [ ]:

# CELL 15: Evaluation Metrics


def calculate_iou(pred, target, num_classes=4):
    """Calculate Intersection over Union per class"""
    ious = []
    pred = pred.view(-1)
    target = target.view(-1)
    
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        
        intersection = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()
        
        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append((intersection / union).item())
    
    return ious

def calculate_f1(pred, target, num_classes=4):
    """Calculate F1 score per class"""
    f1_scores = []
    pred = pred.view(-1)
    target = target.view(-1)
    
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        
        tp = (pred_cls & target_cls).sum().float()
        fp = (pred_cls & ~target_cls).sum().float()
        fn = (~pred_cls & target_cls).sum().float()
        
        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)
        
        f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
        f1_scores.append(f1.item())
    
    return f1_scores

def evaluate_model(model, dataloader, criterion, device):
    """Evaluate model on validation set"""
    model.eval()
    
    total_loss = 0
    all_ious = []
    all_f1s = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            # Forward pass
            outputs = model(images, return_features=False)
            
            # Calculate loss
            loss = criterion(outputs['logits'], masks)
            total_loss += loss.item()
            
            # Get predictions
            preds = outputs['P_x'].argmax(dim=1)
            
            # Calculate metrics
            ious = calculate_iou(preds, masks)
            f1s = calculate_f1(preds, masks)
            
            all_ious.append(ious)
            all_f1s.append(f1s)
    
    # Average metrics
    avg_loss = total_loss / len(dataloader)
    avg_ious = np.nanmean(all_ious, axis=0)
    avg_f1s = np.nanmean(all_f1s, axis=0)
    mean_iou = np.nanmean(avg_ious)
    mean_f1 = np.nanmean(avg_f1s)
    
    return {
        'loss': avg_loss,
        'mean_iou': mean_iou,
        'mean_f1': mean_f1,
        'class_ious': avg_ious,
        'class_f1s': avg_f1s
    }

print("✅ Evaluation functions defined!")

In [ ]:
# ============================================
# FIX: Reduce Memory Usage
# ============================================

# REDUCE BATCH SIZE (most important fix)
config.BATCH_SIZE = 4  # Changed from 8 to 4

# Update NUM_WORKERS for stability
config.NUM_WORKERS = 2

print("="*60)
print("⚙️ UPDATED CONFIGURATION")
print("="*60)
print(f"Batch Size: {config.BATCH_SIZE} (reduced for memory)")
print(f"Num Workers: {config.NUM_WORKERS}")
print("="*60 + "\n")

In [ ]:

# CELL 16: Training Loop (Memory Optimized)


import time
from datetime import datetime
import gc

# Clear memory before training
torch.cuda.empty_cache()
gc.collect()

# Training configuration
num_epochs = 60
best_val_iou = 0.0
patience_counter = 0
early_stop_patience = 15
checkpoint_dir = config.OUTPUT_DIR / 'checkpoints'
checkpoint_dir.mkdir(exist_ok=True)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_iou': [],
    'val_f1': [],
    'lr': []
}

class_names = ['No-Damage', 'Minor', 'Major', 'Destroyed']

print("="*80)
print(f"{'🚀 STARTING TRAINING':^80}")
print("="*80)
print(f"\nConfiguration:")
print(f"  Epochs: {num_epochs}")
print(f"  Batch Size: {config.BATCH_SIZE}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Device: {config.DEVICE}")
print(f"  Checkpoint dir: {checkpoint_dir}")
print(f"  Memory optimizations: Enabled")
print("\n" + "="*80 + "\n")

start_time = time.time()

for epoch in range(1, num_epochs + 1):
    epoch_start = time.time()
    
    # ============ TRAINING ============
    model.train()
    train_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}")
    
    for batch_idx, batch in enumerate(progress_bar):
        images = batch['image'].to(config.DEVICE)
        masks = batch['mask'].to(config.DEVICE)
        
        # Forward pass - Don't extract features during training (saves memory)
        optimizer.zero_grad()
        outputs = model(images, return_features=False)
        
        # Calculate loss
        loss = criterion(outputs['logits'], masks)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        train_loss += loss.item()
        
        # Clear cache periodically
        if batch_idx % 20 == 0:
            torch.cuda.empty_cache()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'lr': f"{optimizer.param_groups[0]['lr']:.2e}",
            'mem': f"{torch.cuda.memory_allocated(0)/1e9:.1f}GB"
        })
    
    avg_train_loss = train_loss / len(train_loader)
    
    # Clear cache before validation
    torch.cuda.empty_cache()
    gc.collect()
    
    # ============ VALIDATION ============
    val_metrics = evaluate_model(model, val_loader, criterion, config.DEVICE)
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_iou'].append(val_metrics['mean_iou'])
    history['val_f1'].append(val_metrics['mean_f1'])
    history['lr'].append(current_lr)
    
    # Print epoch summary
    epoch_time = time.time() - epoch_start
    
    print(f"\n{'='*80}")
    print(f"Epoch {epoch}/{num_epochs} Summary (Time: {epoch_time:.1f}s)")
    print(f"{'='*80}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss:   {val_metrics['loss']:.4f}")
    print(f"  Val mIoU:   {val_metrics['mean_iou']:.4f}")
    print(f"  Val mF1:    {val_metrics['mean_f1']:.4f}")
    print(f"  LR:         {current_lr:.2e}")
    
    print(f"\n  Per-Class IoU:")
    for i, (name, iou) in enumerate(zip(class_names, val_metrics['class_ious'])):
        print(f"    {name:12s}: {iou:.4f}")
    
    print(f"\n  Per-Class F1:")
    for i, (name, f1) in enumerate(zip(class_names, val_metrics['class_f1s'])):
        print(f"    {name:12s}: {f1:.4f}")
    
    # Save best model
    if val_metrics['mean_iou'] > best_val_iou:
        best_val_iou = val_metrics['mean_iou']
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_iou': val_metrics['mean_iou'],
            'val_f1': val_metrics['mean_f1'],
            'val_loss': val_metrics['loss'],
            'class_ious': val_metrics['class_ious'].tolist(),
            'class_f1s': val_metrics['class_f1s'].tolist(),
            'config': {
                'num_classes': config.NUM_CLASSES,
                'F_sat_dim': config.F_SAT_DIM,
                'F_region_dim': config.F_REGION_DIM,
                'image_size': config.IMAGE_SIZE,
                'batch_size': config.BATCH_SIZE
            }
        }
        
        best_path = checkpoint_dir / 'best_model.pth'
        torch.save(checkpoint, best_path)
        print(f"\n  ✅ Best model saved! (IoU: {best_val_iou:.4f})")
    
    # Save checkpoint every 5 epochs
    if epoch % 5 == 0:
        checkpoint_path = checkpoint_dir / f'checkpoint_epoch_{epoch}.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_iou': val_metrics['mean_iou']
        }, checkpoint_path)
        print(f"  💾 Checkpoint saved: {checkpoint_path.name}")
    
    print("="*80 + "\n")
    
    # Early stopping check
    if not (val_metrics['mean_iou'] > best_val_iou):
        patience_counter += 1
        if patience_counter >= early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {early_stop_patience} epochs)")
            break

    # Clear cache after each epoch
    torch.cuda.empty_cache()
    gc.collect()

# Training complete
total_time = time.time() - start_time

print("\n" + "="*80)
print(f"{'✅ TRAINING COMPLETE':^80}")
print("="*80)
print(f"  Total time: {total_time/3600:.2f} hours")
print(f"  Average time per epoch: {total_time/num_epochs/60:.1f} minutes")
print(f"  Best Val IoU: {best_val_iou:.4f}")
print(f"  Best model saved at: {checkpoint_dir / 'best_model.pth'}")
print("="*80 + "\n")

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(config.OUTPUT_DIR / 'training_history.csv', index=False)
print(f"💾 Training history saved: {config.OUTPUT_DIR / 'training_history.csv'}")

In [ ]:

# CELL 17: Plot Training History


import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

epochs_range = range(1, num_epochs + 1)

# Loss plot
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2, markersize=4)
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-s', label='Val Loss', linewidth=2, markersize=4)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# IoU plot
axes[0, 1].plot(epochs_range, history['val_iou'], 'g-o', linewidth=2, markersize=4)
axes[0, 1].axhline(y=best_val_iou, color='r', linestyle='--', linewidth=2, label=f'Best: {best_val_iou:.4f}')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Mean IoU', fontsize=12)
axes[0, 1].set_title('Validation Mean IoU', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, max(history['val_iou']) * 1.1])

# F1 Score plot
axes[1, 0].plot(epochs_range, history['val_f1'], 'm-o', linewidth=2, markersize=4)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Mean F1 Score', fontsize=12)
axes[1, 0].set_title('Validation Mean F1 Score', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, max(history['val_f1']) * 1.1])

# Learning Rate plot
axes[1, 1].plot(epochs_range, history['lr'], 'c-o', linewidth=2, markersize=4)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3, which='both')

plt.suptitle(f'Training History - 18 Epochs (Best IoU: {best_val_iou:.4f})', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"💾 Training history plot saved: {config.OUTPUT_DIR / 'training_history.png'}")

# Print final metrics
print("\n" + "="*60)
print("📊 FINAL TRAINING METRICS")
print("="*60)
print(f"  Final Train Loss:  {history['train_loss'][-1]:.4f}")
print(f"  Final Val Loss:    {history['val_loss'][-1]:.4f}")
print(f"  Final Val IoU:     {history['val_iou'][-1]:.4f}")
print(f"  Final Val F1:      {history['val_f1'][-1]:.4f}")
print(f"  Best Val IoU:      {best_val_iou:.4f}")
print("="*60 + "\n")

In [ ]:
# ============================================
# CELL 18: Load Best Model & Feature Extraction (FIXED)
# ============================================

# Define checkpoint directory
checkpoint_dir = config.OUTPUT_DIR / 'checkpoints'
best_checkpoint_path = checkpoint_dir / 'best_model.pth'

# Load checkpoint with weights_only=False (PyTorch 2.6+ compatibility)
print(f"📂 Loading checkpoint: {best_checkpoint_path.name}\n")
best_checkpoint = torch.load(best_checkpoint_path, 
                             map_location=config.DEVICE, 
                             weights_only=False)  # ← KEY FIX

# Load model weights
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

print("="*60)
print("✅ BEST MODEL LOADED")
print("="*60)
print(f"  Checkpoint: {best_checkpoint_path.name}")
print(f"  Epoch: {best_checkpoint['epoch']}")
print(f"  Val IoU: {best_checkpoint['val_iou']:.4f}")
print(f"  Val F1: {best_checkpoint['val_f1']:.4f}")
print(f"  Val Loss: {best_checkpoint['val_loss']:.4f}")

class_names = ['No-Damage', 'Minor', 'Major', 'Destroyed']

print(f"\n  Per-Class IoU:")
for i, (name, iou) in enumerate(zip(class_names, best_checkpoint['class_ious'])):
    print(f"    {name:<15}: {iou:.4f}")

print(f"\n  Per-Class F1:")
for i, (name, f1) in enumerate(zip(class_names, best_checkpoint['class_f1s'])):
    print(f"    {name:<15}: {f1:.4f}")

print("="*60 + "\n")

# Feature extraction function for multimodal fusion
def extract_features_for_fusion(model, image_tensor, device):
    """
    Extract F_sat, P_x, F_region for multimodal fusion
    
    Args:
        model: Trained DeepLabV3Plus_xBD model
        image_tensor: [B, 3, H, W] or single image
        device: cuda/cpu
    
    Returns:
        dict with:
            - 'F_sat': [B, 512] satellite embedding
            - 'P_x': [B, 4, H, W] damage probabilities
            - 'F_region': [B, K] regional statistics
    """
    model.eval()
    
    with torch.no_grad():
        # Handle single image
        if len(image_tensor.shape) == 3:
            image_tensor = image_tensor.unsqueeze(0)
        
        image_tensor = image_tensor.to(device)
        
        # Extract all features
        outputs = model(image_tensor, return_features=True)
        
        return {
            'F_sat': outputs['F_sat'].cpu(),      # Satellite embedding
            'P_x': outputs['P_x'].cpu(),          # Damage probabilities  
            'F_region': outputs['F_region'].cpu() # Regional statistics
        }

# Test feature extraction
print("🧪 Testing feature extraction...")
sample_batch = next(iter(val_loader))
sample_images = sample_batch['image'][:2].to(config.DEVICE)

features = extract_features_for_fusion(model, sample_images, config.DEVICE)

print(f"\n📊 Extracted Features :")
print(f"{'='*75}")
print(f"{'Feature':<20} {'Symbol':<10} {'Shape':<25} {'Use':<20}")
print(f"{'='*75}")
print(f"{'Satellite embedding':<20} {'F_sat':<10} {str(list(features['F_sat'].shape)):<25} {'Physical damage':<20}")
print(f"{'Damage probs':<20} {'P_x':<10} {str(list(features['P_x'].shape)):<25} {'Severity semantics':<20}")
print(f"{'Region stats':<20} {'F_region':<10} {str(list(features['F_region'].shape)):<25} {'Spatial context':<20}")
print(f"{'='*75}\n")

print("✅ Feature extraction ready for multimodal fusion!")
print("   These features can now be fused with BLIP text/image embeddings")
print("="*60 + "\n")

# Save best val IoU for later cells
best_val_iou = best_checkpoint['val_iou']

# Clear memory
del sample_images, features
torch.cuda.empty_cache()

In [ ]:

# CELL 19: Visualize Predictions


def visualize_predictions(model, dataset, device, num_samples=8):
    """Visualize model predictions vs ground truth"""
    
    model.eval()
    
    # Create figure
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 3))
    
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    class_names_viz = ['No-Damage', 'Minor', 'Major', 'Destroyed']
    
    # Random samples
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    for idx, sample_idx in enumerate(indices):
        sample = dataset[sample_idx]
        
        # Prepare image
        image_tensor = sample['image'].unsqueeze(0).to(device)
        gt_mask = sample['mask'].numpy()
        
        # Get prediction
        with torch.no_grad():
            outputs = model(image_tensor, return_features=False)
            pred_mask = outputs['P_x'].argmax(dim=1).squeeze().cpu().numpy()
        
        # Denormalize image
        img = denormalize_image(sample['image']).permute(1, 2, 0).numpy()
        
        # Calculate accuracy
        accuracy = (pred_mask == gt_mask).mean() * 100
        
        # Plot
        axes[idx, 0].imshow(img)
        axes[idx, 0].set_title(f"Image\n{sample['disaster']}", fontsize=10)
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(gt_mask, cmap='RdYlGn_r', vmin=0, vmax=3)
        axes[idx, 1].set_title("Ground Truth", fontsize=10)
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(pred_mask, cmap='RdYlGn_r', vmin=0, vmax=3)
        axes[idx, 2].set_title(f"Prediction\nAcc: {accuracy:.1f}%", fontsize=10)
        axes[idx, 2].axis('off')
        
        # Overlay
        axes[idx, 3].imshow(img)
        axes[idx, 3].imshow(pred_mask, alpha=0.6, cmap='RdYlGn_r', vmin=0, vmax=3)
        axes[idx, 3].set_title("Overlay", fontsize=10)
        axes[idx, 3].axis('off')
    
    plt.suptitle(f'Model Predictions (Best Val IoU: {best_val_iou:.4f})', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(config.OUTPUT_DIR / 'predictions_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"💾 Predictions saved: {config.OUTPUT_DIR / 'predictions_visualization.png'}")

# Visualize predictions
print("🎨 Generating prediction visualizations...")
visualize_predictions(model, val_dataset, config.DEVICE, num_samples=8)

# Clear memory
torch.cuda.empty_cache()

In [ ]:

# CELL 20: Final Summary & Export


# Comprehensive summary
final_summary = {
    'training': {
        'epochs_completed': num_epochs,
        'best_epoch': best_checkpoint['epoch'],
        'best_val_iou': float(best_checkpoint['val_iou']),
        'best_val_f1': float(best_checkpoint['val_f1']),
        'best_val_loss': float(best_checkpoint['val_loss']),
        'final_train_loss': float(history['train_loss'][-1]),
        'final_val_loss': float(history['val_loss'][-1]),
        'final_val_iou': float(history['val_iou'][-1]),
        'final_val_f1': float(history['val_f1'][-1]),
        'total_training_time_hours': total_time / 3600,
        'avg_epoch_time_minutes': (total_time / num_epochs) / 60
    },
    'model_architecture': {
        'name': 'DeepLabV3Plus_xBD',
        'encoder': 'ResNet101',
        'pretrained': 'ImageNet',
        'total_parameters': total_params,
        'trainable_parameters': trainable_params,
        'model_size_mb': (total_params * 4) / 1e6
    },
    'output_features': {
        'F_sat': {
            'dimension': config.F_SAT_DIM,
            'description': 'Satellite embedding for physical damage',
            'shape': f'[batch, {config.F_SAT_DIM}]'
        },
        'P_x': {
            'dimension': 4,
            'description': 'Damage probability maps (severity semantics)',
            'shape': '[batch, 4, H, W]',
            'classes': ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
        },
        'F_region': {
            'dimension': config.F_REGION_DIM,
            'description': 'Regional statistics (spatial context)',
            'shape': f'[batch, {config.F_REGION_DIM}]'
        }
    },
    'dataset': {
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'total_samples': len(train_df) + len(val_df),
        'image_size': f'{config.IMAGE_SIZE}x{config.IMAGE_SIZE}',
        'batch_size': config.BATCH_SIZE,
        'disaster_types': train_df['disaster'].unique().tolist(),
        'num_disasters': len(train_df['disaster'].unique())
    },
    'class_performance': {
        class_names[i]: {
            'iou': float(best_checkpoint['class_ious'][i]),
            'f1': float(best_checkpoint['class_f1s'][i])
        }
        for i in range(4)
    },
    'training_config': {
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR',
        'loss_function': 'CrossEntropyLoss (class-weighted)',
        'class_weights': class_weights_tensor.tolist(),
        'gradient_clipping': 1.0,
        'data_augmentation': [
            'HorizontalFlip', 'VerticalFlip', 'RandomRotate90',
            'ShiftScaleRotate', 'BrightnessContrast', 'GaussianNoise'
        ]
    },
    'saved_files': {
        'best_model': str(checkpoint_dir / 'best_model.pth'),
        'training_history_csv': str(config.OUTPUT_DIR / 'training_history.csv'),
        'training_history_plot': str(config.OUTPUT_DIR / 'training_history.png'),
        'predictions_visualization': str(config.OUTPUT_DIR / 'predictions_visualization.png'),
        'config': str(config.OUTPUT_DIR / 'training_summary.json')
    }
}

# Save summary
summary_path = config.OUTPUT_DIR / 'training_summary.json'
with open(summary_path, 'w') as f:
    json.dump(final_summary, f, indent=2)

# Print final summary
print("\n" + "="*80)
print(f"{'🎉 TRAINING PIPELINE COMPLETE':^80}")
print("="*80)

print(f"\n📊 FINAL RESULTS:")
print(f"{'─'*80}")
print(f"  Training Time:         {total_time/3600:.2f} hours")
print(f"  Epochs Completed:      {num_epochs}")
print(f"  Best Validation IoU:   {best_checkpoint['val_iou']:.4f}")
print(f"  Best Validation F1:    {best_checkpoint['val_f1']:.4f}")
print(f"  Best Validation Loss:  {best_checkpoint['val_loss']:.4f}")
print(f"  Best Epoch:            {best_checkpoint['epoch']}")

print(f"\n📈 PER-CLASS PERFORMANCE (Best Model):")
print(f"{'─'*80}")
print(f"  {'Class':<15} {'IoU':>10} {'F1':>10}")
print(f"  {'─'*35}")
for i, name in enumerate(class_names):
    iou = best_checkpoint['class_ious'][i]
    f1 = best_checkpoint['class_f1s'][i]
    print(f"  {name:<15} {iou:>10.4f} {f1:>10.4f}")

print(f"\n🎯 MODEL OUTPUTS (Ready for Multimodal Fusion):")
print(f"{'─'*80}")
print(f"  Feature          Symbol      Shape              Use")
print(f"  {'─'*70}")
print(f"  Sat. Embedding   F_sat       [B, {config.F_SAT_DIM}]        Physical damage")
print(f"  Damage Probs     P_x         [B, 4, H, W]     Severity semantics")
print(f"  Region Stats     F_region    [B, {config.F_REGION_DIM}]        Spatial context")

print(f"\n💾 SAVED FILES:")
print(f"{'─'*80}")
print(f"  Best Model:          {checkpoint_dir / 'best_model.pth'}")
print(f"  Training History:    {config.OUTPUT_DIR / 'training_history.csv'}")
print(f"  Training Plot:       {config.OUTPUT_DIR / 'training_history.png'}")
print(f"  Predictions:         {config.OUTPUT_DIR / 'predictions_visualization.png'}")
print(f"  Summary JSON:        {summary_path}")

print(f"\n🔄 NEXT STEPS:")
print(f"{'─'*80}")
print(f"  1. Download best_model.pth for multimodal fusion")
print(f"  2. Use extract_features_for_fusion() to get F_sat, P_x, F_region")
print(f"  3. Fuse with BLIP embeddings for complete multimodal system")
print(f"  4. Save version to preserve all trained weights")

print("\n" + "="*80)
print(f"{'✅ DEEPLABV3+ TRAINING COMPLETE!':^80}")
print("="*80)



In [ ]:
batch = next(iter(val_loader))

# Print keys once to confirm
print("Batch keys:", batch.keys())

images = batch[list(batch.keys())[0]].to(device)
masks  = batch[list(batch.keys())[1]]


In [ ]:
# ============================================
# FINAL FEATURE VISUALIZATION CELL
# ============================================

import matplotlib.pyplot as plt
import numpy as np

model.eval()

# Get batch
batch = next(iter(val_loader))

images = batch['image'].to(config.DEVICE)
masks = batch['mask']
image_ids = batch['image_id']
disasters = batch['disaster']

with torch.no_grad():
    outputs = model(images, return_features=True)

# Extract outputs
logits = outputs['logits']
P_x = outputs['P_x']              # Damage probabilities
F_sat = outputs['F_sat']          # Satellite embedding
F_region = outputs['F_region']    # Regional stats

pred_mask = torch.argmax(P_x, dim=1)

# Select first image in batch
idx = 0

img = images[idx].cpu().permute(1,2,0).numpy()
mask_pred = pred_mask[idx].cpu().numpy()
probs_map = P_x[idx].cpu().numpy()

sat_embedding = F_sat[idx].cpu().numpy()
region_vector = F_region[idx].cpu().numpy()

image_id = image_ids[idx]
disaster_type = disasters[idx]

# Compute mean class probabilities
mean_probs = probs_map.mean(axis=(1,2))

# ============================================
# VISUALIZATION
# ============================================

plt.figure(figsize=(18,6))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title(f"Satellite Image\nID: {image_id}\nDisaster: {disaster_type}")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(mask_pred, cmap="jet")
plt.title("Predicted Damage Mask")
plt.axis("off")

plt.subplot(1,3,3)
plt.bar(["No-Damage","Minor","Major","Destroyed"], mean_probs)
plt.title("Mean Damage Probabilities")

plt.show()

# ============================================
# PRINT FEATURE OUTPUTS
# ============================================

print("\n" + "="*70)
print("📊 EXTRACTED FEATURES (For This Image)")
print("="*70)

print(f"\n🛰 F_sat (Satellite Embedding)")
print("Shape:", sat_embedding.shape)
print("First 10 values:", sat_embedding[:10])

print(f"\n📈 F_region (Regional Statistics Vector)")
print("Shape:", region_vector.shape)
print("First 10 values:", region_vector[:10])

print(f"\n📊 P_x (Mean Damage Probabilities)")
print(f"No-Damage : {mean_probs[0]:.4f}")
print(f"Minor     : {mean_probs[1]:.4f}")
print(f"Major     : {mean_probs[2]:.4f}")
print(f"Destroyed : {mean_probs[3]:.4f}")

print("\n" + "="*70)
print("✅ These are the exact multimodal-ready features")
print("   F_sat → Physical damage embedding")
print("   P_x → Severity semantics")
print("   F_region → Spatial context representation")
print("="*70)
